# Preprocesado PhysioNet Challenge 2018
### Fragmentacion del sueno -- EEG C3-M2 y C4-M1

**Output por paciente:** un `.npz` con:
- `signals` : `(N, 2, 6000)` -- senal filtrada, 2 canales x 30 s
- `psd`     : `(N, 2, 159)`  -- PSD Welch, 0.5-40 Hz
- `labels`  : `(N,)`         -- 0 = non-arousal, 1 = arousal
- `freqs`   : `(159,)`       -- eje frecuencial


In [8]:
"""Notebook driver: pipeline lives in ``src/process.py``. Editable install: ``pip install -e .``"""
from pathlib import Path
import sys

for _root in (Path.cwd(), *Path.cwd().parents):
    if (_root / "pyproject.toml").exists():
        _src = _root / "src"
        if _src.is_dir() and str(_src) not in sys.path:
            sys.path.insert(0, str(_src))
        break

import numpy as np

from process import (
    PreprocessConfig,
    discover_patient_roots,
    print_batch_summary,
    run_batch,
)

# === CONFIGURACION -- edita aqui ===================================
DATA_DIR = Path("Data")
# OUT_DIR = None  # guarda *_preprocessed.npz junto a cada .mat
OUT_DIR = Path("Processed")
cfg = PreprocessConfig()
# ====================================================================

print("Configuracion cargada")
print(f"  Data dir : {DATA_DIR.resolve()}")
print(
    "  Out dir  : "
    + (str(OUT_DIR.resolve()) if OUT_DIR is not None else "guardado junto a cada paciente (.npz)")
)


Configuracion cargada
  Data dir : c:\Users\45125164S\Desktop\UPC\mHealth\Project\Data
  Out dir  : c:\Users\45125164S\Desktop\UPC\mHealth\Project\Processed


In [9]:
# Implemented in ``src/process.py``.


Funciones definidas OK


In [10]:
patient_bases = discover_patient_roots(DATA_DIR)
print(f"Pacientes encontrados: {len(patient_bases)}")
for b in patient_bases:
    print(f"  {b}")


Pacientes encontrados: 11
  Data\0005\tr03-0005
  Data\0029\tr03-0029
  Data\0052\tr03-0052
  Data\0061\tr03-0061
  Data\0078\tr03-0078
  Data\0079\tr03-0079
  Data\0083\tr03-0083
  Data\0086\tr03-0086
  Data\0087\tr03-0087
  Data\0092\tr03-0092
  Data\0100\tr03-0100


In [11]:
if OUT_DIR is not None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

summary = run_batch(DATA_DIR, OUT_DIR, cfg=cfg)


Procesando 11 paciente(s)...

  [tr03-0005] Cargando... 5,147,000 muestras -> 857 ventanas -> 423 validas  (arousal=30, non-arousal=393, ratio=7.1%)
  -> Guardado: Processed\tr03-0005_preprocessed.npz  (19.4 MB)

  [tr03-0029] Cargando... 4,770,000 muestras -> 795 ventanas -> 294 validas  (arousal=41, non-arousal=253, ratio=13.9%)
  -> Guardado: Processed\tr03-0029_preprocessed.npz  (13.5 MB)

  [tr03-0052] Cargando... 5,992,000 muestras -> 998 ventanas -> 464 validas  (arousal=1, non-arousal=463, ratio=0.2%)
  -> Guardado: Processed\tr03-0052_preprocessed.npz  (21.3 MB)

  [tr03-0061] Cargando... 5,236,000 muestras -> 872 ventanas -> 570 validas  (arousal=19, non-arousal=551, ratio=3.3%)
  -> Guardado: Processed\tr03-0061_preprocessed.npz  (26.1 MB)

  [tr03-0078] Cargando... 5,840,000 muestras -> 973 ventanas -> 604 validas  (arousal=35, non-arousal=569, ratio=5.8%)
  -> Guardado: Processed\tr03-0078_preprocessed.npz  (27.7 MB)

  [tr03-0079] Cargando... 5,363,000 muestras -> 893 ven

In [12]:
print_batch_summary(summary)


Paciente        Ventanas   Arousal    Non-ar   Ratio%     MB
------------------------------------------------------------
  tr03-0005          423        30       393     7.1%  19.4
  tr03-0029          294        41       253    13.9%  13.5
  tr03-0052          464         1       463     0.2%  21.3
  tr03-0061          570        19       551     3.3%  26.1
  tr03-0078          604        35       569     5.8%  27.7
  tr03-0079          706         7       699     1.0%  32.4
  tr03-0083          414        27       387     6.5%  19.0
  tr03-0086          686        35       651     5.1%  31.1
  tr03-0087          813         8       805     1.0%  37.2
  tr03-0092          623        86       537    13.8%  28.6
  tr03-0100          488       285       203    58.4%  22.3
------------------------------------------------------------
  TOTAL             6085       574      5511     9.4%

Pesos sugeridos para weighted CrossEntropyLoss:
  weight = torch.tensor([1.0, 9.60])  # [non-arousal, 

In [13]:
# Verificacion rapida del primer .npz generado
if summary:
    d = np.load(summary[0]['path'], allow_pickle=True)
    pid = summary[0]['patient']
    print(f"Verificacion: {pid}")
    print(f"  signals : {d['signals'].shape}  dtype={d['signals'].dtype}")
    print(f"  psd     : {d['psd'].shape}  dtype={d['psd'].dtype}")
    print(f"  labels  : {d['labels'].shape}  valores={np.unique(d['labels'])}")
    print(f"  freqs   : {d['freqs'].shape}  rango={d['freqs'][0]:.2f}-{d['freqs'][-1]:.2f} Hz")
    print(f"  canales : {list(d['ch_names'])}")
    print()
    print("Ejemplo de carga en PyTorch:")
    print("  import torch")
    print("  signals = torch.tensor(d['signals'])  # (N, 2, 6000)  -> CNN 1D")
    print("  psd     = torch.tensor(d['psd'])      # (N, 2, 159)   -> CNN espectral")
    print("  labels  = torch.tensor(d['labels'], dtype=torch.long)  # (N,)")


Verificacion: tr03-0005
  signals : (423, 2, 6000)  dtype=float32
  psd     : (423, 2, 159)  dtype=float32
  labels  : (423,)  valores=[0 1]
  freqs   : (159,)  rango=0.50-40.00 Hz
  canales : [np.str_('C3-M2'), np.str_('C4-M1')]

Ejemplo de carga en PyTorch:
  import torch
  signals = torch.tensor(d['signals'])  # (N, 2, 6000)  -> CNN 1D
  psd     = torch.tensor(d['psd'])      # (N, 2, 159)   -> CNN espectral
  labels  = torch.tensor(d['labels'], dtype=torch.long)  # (N,)
